# 00 — Data and Connections through the UI

[Guide](../README.md) · [00: Data and Connections](00_setup_and_data.ipynb) · [01: Create an experiment](01_create_experiment.ipynb) · [02: Review results](02_review_results.ipynb) · [03: Deploy and predict](03_deploy_and_predict.ipynb)

[Equivalent code notebook](../00_prepare_data.ipynb)

Follow this route to run the same fraud-detection story through dashboard forms: **choose data → train → inspect results → deploy → score Sally's transactions**. These four notebooks are a readable walkthrough; they contain no executable cells and do not submit anything when opened.

You will use the RHOAI dashboard, an object-storage web console if uploads are needed, and a terminal with `curl` for the final prediction (or a graphical HTTP client such as Postman). `.env` and the Python helpers are only needed if you choose the existing code route. Menu labels can vary with the installed RHOAI version; use the screenshots captured for your cluster as the visual reference.

**Dataset input in this release:** provide only the training CSV, such as `train.csv`. Separate evaluation/test data cannot be supplied. That capability is planned for later releases; evaluation is handled automatically by the managed pipeline.

## 1. Create a new project

1. Sign in to the RHOAI dashboard and open **Projects** (called **Data science projects** in some releases).
2. Choose **Create project**. Enter a display name such as `Fraud Detection AutoML`, add an optional description, and create the project.
3. Open the newly created project and record its display name and Kubernetes namespace in the worksheet below. The namespace can differ from the display name; use the actual namespace shown by the project details.
4. Keep this project selected for every Connection, pipeline, experiment, and deployment in this walkthrough. If you cannot create a project, ask your administrator for the required access.

The project is new: you will add storage Connections next, and only then configure its pipeline server and select the managed pipeline definition.

For this route you do not need to run the Keras, ONNX, Ray, Training Operator, or Feast chapters. A workbench is needed only if you choose code-based preparation or open generated notebooks for local prediction.

<!-- SCREENSHOT UI-01: replace screenshots/00_1_create_project.png; keep the filename. -->

<a href="screenshots/00_1_create_project.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/00_1_create_project.png" alt="UI-01: Create a new project" width="1500" style="max-width: 100%; height: auto;">
</a>

## 2. Add storage Connections to the new project

1. Open **Connections** in the project you just created and choose **Create connection** (or **Add connection** on your release).
2. Select the S3-compatible Connection type and enter a descriptive name, the storage endpoint, region, bucket, and credentials. The bucket must already exist and be reachable by pipeline pods. Save the Connection.
3. Record its display name and Kubernetes Secret name. Select this Connection in AutoML by its displayed name; keep the Secret name as a reference if you later use the code route.
4. Decide which bucket the pipeline server will use for output artifacts. If it is the input bucket, reuse this Connection. If it is a different bucket, add a second S3 Connection for that artifact bucket now. You will use it in the pipeline-server configuration and later for deployment.

At this point you are creating Connections, not running a pipeline. Configure the pipeline server in the next section, after the Connections exist. Keep credentials in the Connection forms; do not copy them into this notebook or screenshots.

<!-- SCREENSHOT UI-02: replace screenshots/00_2_storage_connections.png; keep the filename. -->

<a href="screenshots/00_2_storage_connections.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/00_2_storage_connections.png" alt="UI-02: Input and artifact Connections" width="1500" style="max-width: 100%; height: auto;">
</a>

## 3. Configure the pipeline server and select its managed pipeline definition

### Configure the server using the artifact Connection

1. In the same new project, open **Pipelines** and choose **Configure pipeline server**.
2. Use **Autofill from connection**, or the equivalent storage selector, to select the Connection for the **artifact bucket** created in section 2. Check the endpoint, region, and bucket populated by the form.
3. Review the remaining server settings for your cluster and save the configuration. Wait until the pipeline server is ready before continuing.
4. Record the actual artifact bucket in the worksheet. The server's output storage is independent of the training input object path you will choose later.

<!-- SCREENSHOT UI-02A: replace screenshots/00_3_pipeline_server.png; keep the filename. -->

<a href="screenshots/00_3_pipeline_server.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/00_3_pipeline_server.png" alt="UI-02A: Configure the pipeline server" width="1500" style="max-width: 100%; height: auto;">
</a>

### Select the AutoML pipeline definition

1. Open the project's available pipelines/managed pipelines and locate the installed GA definition **`autogluon-tabular-training-pipeline`**.
2. Open its details and select or record the installed version you intend to use. This is the pipeline definition for the tabular experiment, not an ONNX training pipeline or an uploaded model artifact.
3. Review the installed definition's training, task, preset, and evaluation settings. The current AutoML input is the training CSV only. Evaluation is handled by the managed pipeline; there is no separate evaluation dataset to configure in this walkthrough.
4. If the GA definition is not registered, ask the administrator to enable/register a compatible managed definition. This walkthrough does not ask you to compile or import a custom pipeline.
5. Confirm that the cluster also offers a compatible AutoGluon serving runtime for the deployment stage.

Do not submit a run yet. First prepare the training object in section 4, then use [01 — Create an experiment](01_create_experiment.ipynb) to select its path and submit once.

<!-- SCREENSHOT UI-02B: replace screenshots/00_4_managed_pipeline_definition.png; keep the filename. -->

<a href="screenshots/00_4_managed_pipeline_definition.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/00_4_managed_pipeline_definition.png" alt="UI-02B: Managed pipeline definition" width="1500" style="max-width: 100%; height: auto;">
</a>

## 4. Choose the training file

The training CSV must have the seven numeric columns below plus the binary label `fraud`, with both 0 and 1 present:

`distance_from_home`, `distance_from_last_transaction`, `ratio_to_median_purchase_price`, `repeat_retailer`, `used_chip`, `used_pin_number`, `online_order`.

Choose **one** preparation option:

| Option | Training input | When to use it |
| --- | --- | --- |
| Reuse prepared training data | The train S3 URI recorded by [code notebook 00](../00_prepare_data.ipynb), usually train + validate | Reuse the same training object without another upload |
| Prepare through the storage UI | Upload the repository's `data/train.csv` (600,000 rows) | Complete the native UI route without executing Python; leave `validate.csv` unused because AutoML creates internal splits |
| Use your own CSV | Your labeled training file | Follow the same schema and record its actual count |

For preparation through the storage console (for example, AWS S3 or MinIO), select the input bucket and create a fresh prefix such as `fraud-detection/automl/ui/<YOUR_DATASET_ID>/`. Upload the chosen training CSV as `train.csv`. Wait for the upload to finish and check its object size.

**There is no external test dataset upload/selection in the current AutoML wizard.** Do not upload `data/test.csv` for this native UI experiment or merge it into training. The pipeline derives evaluation data internally from the supplied training input. External-test selection is planned for a later release.

The merged code-route training set has 800,000 rows, while the original `train.csv` has 600,000. Reusing the training object aligns the training input, but the code route's external test and the UI route's internal evaluation still differ. Do not compare their scores as if the evaluation data were identical. This UI walkthrough supplies only the training dataset and records the metrics produced automatically by the pipeline.

<!-- SCREENSHOT UI-03: replace screenshots/00_5_training_data_upload.png; keep the filename. -->

<a href="screenshots/00_5_training_data_upload.png" target="_blank" rel="noopener noreferrer">
  <img src="screenshots/00_5_training_data_upload.png" alt="UI-03: Training object in storage" width="1500" style="max-width: 100%; height: auto;">
</a>

## 5. Record the inputs for the next notebook

In Jupyter, double-click this Markdown cell to fill in the worksheet. In a read-only viewer, copy the table into your own notes. Use these exact values in the experiment and deployment forms. Record paths only; keep credentials in the Connections.

| Item | Your value |
| --- | --- |
| Dashboard base URL | `https://<YOUR_DASHBOARD_HOST>` |
| Project display name / namespace | `<YOUR_PROJECT_NAME>` / `<YOUR_PROJECT>` |
| Input Connection display name / Secret name | `<YOUR_CONNECTION>` / `<YOUR_CONNECTION_SECRET>` |
| Input bucket | `<YOUR_INPUT_BUCKET>` |
| Training object key (no bucket or s3://) | `fraud-detection/automl/ui/<YOUR_DATASET_ID>/train.csv` |
| Actual training rows / fraud rate | `<TRAIN_ROWS>` / `<TRAIN_FRAUD_RATE>` |
| Evaluation source | `Internal split of training input (native AutoML UI)` |
| Pipeline artifact bucket | `<YOUR_ARTIFACT_BUCKET>` |
| Artifact Connection | `<YOUR_ARTIFACT_CONNECTION>` |
| Managed pipeline definition | `autogluon-tabular-training-pipeline` |
| Selected pipeline version | `<PIPELINE_VERSION_ID>` |

For the original repository training file, the fraud rate is approximately **8.7558%** in 600,000 rows. Merging train + validate gives **8.7525%** in 800,000 rows. Use the actual statistics of your chosen training file. Do not copy `data/test.csv` row counts or fraud rates into the internal evaluation results; that file is not used by the native UI experiment.

**Ready to continue:** the new project exists, its Connections are saved, the pipeline server is ready, the managed definition/version is identified, and the training CSV exists at the recorded location. UI-created experiments will not automatically create the code route's `outputs/run.json`. Continue with the UI notebooks for this run.

[Next: Create an experiment](01_create_experiment.ipynb) · [Return to the guide](../README.md)